# Salary Prediction Model
Salary_Data.csv veri seti üzerinde Ridge Regression, Random Forest ve HistGradientBoosting modellerinin karşılaştırması.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Veri Yükleme
url = "https://raw.githubusercontent.com/OmarAghayev/03.11.2026-class/main/Salary_Data.csv"
df = pd.read_csv(url)

# 2. Eksik Veri ve Kopya Satır Temizliği
df = df.dropna(subset=["Salary"]).reset_index(drop=True)
df = df.drop_duplicates().reset_index(drop=True)

# 3. Eğitim Seviyesi Kodlaması (Ordinal Mapping)
edu_map = {
    "High School": 0,
    "Bachelor's": 1,
    "Bachelor's Degree": 1,
    "Master's": 2,
    "Master's Degree": 2,
    "PhD": 3,
    "phd": 3
}
df["Education_Clean"] = df["Education Level"].map(edu_map).fillna(1)

# Özellikler ve Hedef Değişken
features = ["Age", "Years of Experience", "Education_Clean", "Job Title"]
X = df[features]
y = df["Salary"]

# 4. Train / Test Ayrımı (%20 Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Ön İşleme (StandardScaler ve OneHotEncoder)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Years of Experience", "Education_Clean"]),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Job Title"])
    ]
)

# 6. Modellerin Tanımlanması
models = {
    "Ridge Regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=5, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_depth=6, random_state=42)
}

print(f"Temiz Veri Satır Sayısı: {len(df)}")
print(f"Eğitim Seti: {len(X_train)}, Test Seti: {len(X_test)}\n")

# 7. Model Eğitimi ve Değerlendirme
for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    print(f"--- {name} ---")
    print(f"R2 Score: {r2:.4f}")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}\n")
